[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb)

# 🟠 Medium: Top-k / Top-p (Nucleus) Sampling

Реализуйте **sampling с temperature, top-k и top-p filtering**. Модель выдаёт logits для всего vocabulary, фильтры ограничивают набор кандидатов, а финальный token всё равно выбирается случайно из оставшегося распределения.

### Temperature
`logits / temperature` меняет резкость distribution до Softmax. Маленькая положительная temperature усиливает различия и приближает выбор к argmax; большая делает distribution более плоским. Temperature должна быть больше нуля.

### Top-k
Оставляет ровно `k` token с наибольшими logits. Остальным присваивается `-inf`, чтобы после Softmax их probability стала нулевой. `top_k=1` эквивалентен greedy choice, а `top_k=0` в контракте означает отсутствие этого фильтра.

### Top-p / nucleus
Top-p выбирает минимальный prefix token, отсортированных по убыванию probability, чья cumulative probability достигает порога `p`. Важно сохранить первый token, который пересёк threshold: маску `cumsum > p` обычно сдвигают на одну позицию вправо. Затем маска из sorted order переносится обратно к исходным vocabulary indices.

### Контракт
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) unnormalized log-probabilities
    # Returns: sampled token index
```

### План реализации
1. Масштабируйте logits по temperature.
2. Примените top-k mask, если `top_k > 0`.
3. Для top-p отсортируйте logits, получите probabilities и cumulative sum.
4. Постройте сдвинутую mask и занулите probability исключённых token через `-inf` logits.
5. Выполните Softmax и `multinomial` sampling, верните Python `int` исходного token index.

### Порядок операций
Temperature применяется первой. Top-k и top-p затем работают с уже масштабированными logits; после фильтрации Softmax перенормирует probability оставшихся token до суммы `1`.

### Быстрые самопроверки
- `top_k=1` всегда возвращает argmax;
- очень низкая temperature почти всегда выбирает лучший token;
- без filtering при равных logits со временем встречаются все token;
- результат лежит в `[0,V)`;
- при фиксированном seed sampling воспроизводим.

### Типичные ошибки
- Softmax вычисляется до temperature;
- top-p cumulative sum строится в исходном, а не sorted порядке;
- token, пересекающий `p`, ошибочно удаляется;
- sorted position возвращается как vocabulary index;
- после masking не выполняется перенормировка;
- функция возвращает tensor вместо `int`.

### Официальные материалы и примеры
- [`torch.topk`](https://docs.pytorch.org/docs/stable/generated/torch.topk.html) — top-k values и original indices;
- [`torch.sort`](https://docs.pytorch.org/docs/stable/generated/torch.sort.html) и [`torch.cumsum`](https://docs.pytorch.org/docs/stable/generated/torch.cumsum.html) — nucleus ordering и cumulative mass;
- [`torch.multinomial`](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html) — sampling token index из probability vector.

### Вопросы для самопроверки
1. Почему masked logits задают как `-inf`, а не `0`?
2. Чем фиксированный top-k отличается от динамического nucleus?
3. Зачем top-p mask сдвигают на одну позицию?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
    pass  # temperature, top-k filter, top-p filter, sample

In [ ]:
# 🧪 Debug
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('topk_sampling')